# NumPy, Clearly Explained

### A rewritten, student-friendly version of the Week 4 lecture notebook

> **中文说明**：这是 WEEK4 课件（NumPy 数组）的重写版。原课件内容没有大问题，但顺序乱、解释少、还有几处会直接报错的例子。这一版按 “先看懂 → 再动手 → 再练习” 的顺序重排，补上解释、修正错误，并在关键处加了中文提示。**每个代码块都真实运行过，下面的输出就是运行结果。**

**How to use this notebook**

1. 先读文字说明 → 再看代码 → 最后遮住答案自己做 `Exercise`。
2. 每个新概念都紧跟一个最小例子，尽量一个代码块只讲一件事。
3. 需要 “避坑” 的地方用 ⚠️ 标出。

## Contents

1. Why NumPy?（为什么要用 NumPy）
2. Creating arrays（创建数组）
3. Array attributes（数组属性）
4. Indexing（索引）
5. Slicing（切片）
6. Boolean / conditional indexing（布尔索引）
7. Views vs. copies（视图 vs. 副本）⚠️ 高频考点
8. Reshaping（改变形状）
9. Concatenating & splitting（拼接与拆分）
10. Iterating over arrays（遍历）
11. Element-wise operations & dot product（逐元素运算与点积）
12. Transpose & dtype casting（转置与类型转换）
13. Math / aggregation functions（数学与聚合函数）⚠️ axis 是难点
14. Sorting（排序）
15. Cheat sheet & common pitfalls（速查表与常见坑）

## 1. Why NumPy?（为什么要用 NumPy）

- Python 自带的 `list` 很灵活（能装不同类型），但**没有为数值计算优化**，元素是分散存放在内存里的。
- 标准库 `array` 模块提供了“同类型、连续内存”的数组，但功能很少。
- `numpy.ndarray` 是 NumPy 的核心：**同一种 dtype + 连续内存 + 支持向量化运算**，因此又快又省内存。

一句话：**列表适合“装东西”，NumPy 数组适合“算数学”。**

In [1]:
# Plain Python list -- 灵活，但数学运算行为奇怪
L = list(range(5))
print("list L        :", L)
print("L + L (拼接)   :", L + L)
print("L * 2 (重复)   :", L * 2)

list L        : [0, 1, 2, 3, 4]
L + L (拼接)   : [0, 1, 2, 3, 4, 0, 1, 2, 3, 4]
L * 2 (重复)   : [0, 1, 2, 3, 4, 0, 1, 2, 3, 4]


In [2]:
# 标准库 array：有类型，但功能非常基础
import array

L = list(range(10))
a = array.array('i', L)     # 'i' = signed int
print(a)

array('i', [0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


In [3]:
# NumPy array：数值计算的主角
import numpy as np

b = np.array([1, 2, 3, 4, 5])
print(b)
print("type:", type(b))

[1 2 3 4 5]
type: <class 'numpy.ndarray'>


### 关键区别：同样的运算符，含义完全不同

| Expression | Python list | NumPy array |
|---|---|---|
| `L * 3` / `b * 3` | 把列表重复 3 遍 | 每个元素乘 3（逐元素） |
| `L + L` / `b + b` | 把列表拼起来 | 每个元素相加（逐元素） |

> **中文提示**：这就是 NumPy 的核心价值 —— 同一个 `+` `*`，NumPy 做的是“逐元素数学运算”，而 list 做的是“拼接 / 重复”。

In [4]:
L = [1, 2, 3]
b = np.array([1, 2, 3])

print("list  L * 3 ->", L * 3)     # 重复
print("array b * 3 ->", b * 3)     # 逐元素相乘
print("list  L + L ->", L + L)     # 拼接
print("array b + b ->", b + b)     # 逐元素相加

list  L * 3 -> [1, 2, 3, 1, 2, 3, 1, 2, 3]
array b * 3 -> [3 6 9]
list  L + L -> [1, 2, 3, 1, 2, 3]
array b + b -> [2 4 6]


## 2. Creating arrays（创建数组）

### 2.1 From Python sequences（从 Python 序列创建）

`np.array()` 会**自动推断 dtype**（元素类型）。一个数组里只能有一种 dtype。

In [5]:
print(np.array([1, 2, 3, 4, 5]))                      # 整数 -> int64 (64 位系统)
print(np.array([np.pi, 4, 2, 3]))                      # 有小数 -> float64
print(np.array([np.pi, 4, 2, 3], dtype='float32'))     # 强制用 float32 (更省内存)
print(np.array([1, 2, 3, 4], dtype='float32'))

[1 2 3 4 5]
[3.14159265 4.         2.         3.        ]
[3.1415927 4.        2.        3.       ]
[1. 2. 3. 4.]


In [6]:
# 用嵌套列表创建二维数组
np.array([range(i, i + 3) for i in [2, 4, 6]])

array([[2, 3, 4],
       [4, 5, 6],
       [6, 7, 8]])

⚠️ **混合类型会“向上提升”为同一种类型。** 数字 + 字符串 → 全部变成字符串。

In [7]:
mixed = np.array([1, 2, 'a'])
print(mixed, "| dtype:", mixed.dtype)

['1' '2' 'a'] | dtype: <U21


### 2.2 Built-in constructors（内置构造函数，从零创建）

| Function | Meaning |
|---|---|
| `np.zeros(shape, dtype=...)` | 全 0 |
| `np.ones(shape, dtype=...)`  | 全 1 |
| `np.full(shape, value)`      | 全部填充同一个值 |
| `np.eye(n)`                  | n×n 单位矩阵（对角线为 1） |
| `np.arange(start, stop, step)` | 等差数列，**不含 stop** |
| `np.linspace(start, stop, n)`  | n 个等距点，**包含两端** |

In [8]:
print(np.zeros(10, dtype=int))          # 长度 10 的整数 0
print(np.ones((3, 5), dtype=float))     # 3x5 的浮点 1
print(np.full((3, 5), 3.14))            # 3x5 全部填 3.14
print(np.eye(3))                        # 3x3 单位矩阵

[0 0 0 0 0 0 0 0 0 0]
[[1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]]
[[3.14 3.14 3.14 3.14 3.14]
 [3.14 3.14 3.14 3.14 3.14]
 [3.14 3.14 3.14 3.14 3.14]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


`np.arange` 像 `range`，但返回数组；`np.linspace` 用“点的个数”控制，且包含末尾。

In [9]:
print("arange :", np.arange(0, 10, 1))    # 不含 10
print("range  :", list(range(0, 10)))      # 内建 range，同样不含 10
print("linspace:", np.linspace(0, 1, 5))  # 5 个点，包含 0 和 1

arange : [0 1 2 3 4 5 6 7 8 9]
range  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
linspace: [0.   0.25 0.5  0.75 1.  ]


In [10]:
print(np.array([1, 2, 3] * 3))    # list 重复 -> 再转数组
print(np.repeat([1, 2, 3], 3))    # NumPy 的 repeat：每个元素重复 3 次

[1 2 3 1 2 3 1 2 3]
[1 1 1 2 2 2 3 3 3]


In [11]:
x = np.array([[1, 2], [3, 4]])
print("axis=0 (每一行重复 [1,2] 次):\n", np.repeat(x, [1, 2], axis=0))
print("axis=1 (每一列重复 [1,2] 次):\n", np.repeat(x, [1, 2], axis=1))

axis=0 (每一行重复 [1,2] 次):
 [[1 2]
 [3 4]
 [3 4]]
axis=1 (每一列重复 [1,2] 次):
 [[1 2 2]
 [3 4 4]]


> **中文提示**：`axis=0` 表示“行方向”（上下、竖直），`axis=1` 表示“列方向”（左右、水平）。第 13 节还会反复用到。

### 2.3 NumPy data types（数据类型速查）

| Type | Description |
|---|---|
| `bool_` | Boolean, 1 byte |
| `int_` | Default integer (usually `int64` on 64-bit) |
| `intc` / `intp` | C int / index integer |
| `int8 / int16 / int32 / int64` | Signed integers of 8/16/32/64 bits |
| `uint8 / uint16 / uint32 / uint64` | Unsigned integers |
| `float_` | Shorthand for `float64` |
| `float16 / float32 / float64` | 半 / 单 / 双精度浮点 |
| `complex_` | Shorthand for `complex128` |
| `complex64 / complex128` | 复数（两个 32-bit / 64-bit 浮点） |

> **要点**：默认整数是 `int64`，默认浮点是 `float64`。想要省内存（比如做深度学习）时才显式指定 `float32`。

## 3. Array attributes（数组属性）

用属性就能查看数组的维度、形状、元素个数和内存占用，不需要靠记忆。

In [12]:
x1 = np.ones(4, dtype=float)            # 1-D
x2 = np.ones((3, 4), dtype='float32')   # 2-D
x3 = np.ones((2, 3, 4), dtype='float64')  # 3-D

print("x3.ndim    :", x3.ndim)       # 维度数
print("x3.shape   :", x3.shape)      # 每个维度的大小
print("x3.size    :", x3.size)       # 元素总数
print("x3.dtype   :", x3.dtype)      # 元素类型
print("x3.itemsize:", x3.itemsize, "bytes")   # 每个元素占多少字节
print("x3.nbytes  :", x3.nbytes, "bytes")     # 总字节数
print("size * itemsize == nbytes ?", x3.size * x3.itemsize == x3.nbytes)

x3.ndim    : 3
x3.shape   : (2, 3, 4)
x3.size    : 24
x3.dtype   : float64
x3.itemsize: 8 bytes
x3.nbytes  : 192 bytes
size * itemsize == nbytes ? True


## 4. Indexing（索引）

- 一维：`x[i]`，支持负数（`-1` 是最后一个）。
- 二维：`x[row, col]`，用**逗号**分开两个下标。
- 数组是**可变的（mutable）**，可以直接改某个位置的值。

In [13]:
x1 = np.array([1, 2, 3, 4, 5])
print("x1[0] =", x1[0])
print("x1[3] =", x1[3])
print("x1[-1]=", x1[-1])
print("x1[-2]=", x1[-2])

x1[0] = 1
x1[3] = 4
x1[-1]= 5
x1[-2]= 4


In [14]:
x2 = np.array([[1, 2, 3, 4],
               [5, 6, 7, 8]])
print(x2)
print("x2[0, 0] =", x2[0, 0])     # 第 0 行第 0 列
print("x2[0, -1]=", x2[0, -1])    # 第 0 行最后一列

[[1 2 3 4]
 [5 6 7 8]]
x2[0, 0] = 1
x2[0, -1]= 4


In [15]:
x2[0, 0] = 9       # 修改元素
print(x2)

[[9 2 3 4]
 [5 6 7 8]]


⚠️ **dtype 不会因为赋值而改变。** 往整数数组里塞小数，小数会被直接截断。

In [16]:
x1 = np.array([1, 2, 3, 4, 5])   # int 数组
x1[0] = np.pi                    # 3.14159... 塞不进 int
print(x1, "| dtype:", x1.dtype)  # 结果是 3，不是 3.14

[3 2 3 4 5] | dtype: int64


## 5. Slicing（切片）

黄金法则：`x[start : stop : step]`

- `start` 省略 → 从 0 开始
- `stop` 省略 → 到最后
- **`stop` 不包含**（左闭右开）
- `step` 为负 → 反向

In [17]:
x = np.arange(0, 10)
print("x        :", x)
print("x[:5]    :", x[:5])      # 前 5 个 (0..4)
print("x[5:]    :", x[5:])      # 从 index 5 到最后
print("x[4:7]   :", x[4:7])     # start:stop (不含 7)
print("x[::2]   :", x[::2])     # 每隔一个
print("x[1::2]  :", x[1::2])    # 每隔一个，从 1 开始
print("x[1::3]  :", x[1::3])    # 每隔三个，从 1 开始

x        : [0 1 2 3 4 5 6 7 8 9]
x[:5]    : [0 1 2 3 4]
x[5:]    : [5 6 7 8 9]
x[4:7]   : [4 5 6]
x[::2]   : [0 2 4 6 8]
x[1::2]  : [1 3 5 7 9]
x[1::3]  : [1 4 7]


In [18]:
print("x[:-1]   :", x[:-1])     # 去掉最后一个
print("x[::-1]  :", x[::-1])    # 整个反转
print("x[5::-1] :", x[5::-1])   # 从 index 5 往回
print("x[5::-2] :", x[5::-2])   # 从 index 5 往回，每隔一个
print("x[7::-3] :", x[7::-3])   # 从 index 7 往回，每隔三个

x[:-1]   : [0 1 2 3 4 5 6 7 8]
x[::-1]  : [9 8 7 6 5 4 3 2 1 0]
x[5::-1] : [5 4 3 2 1 0]
x[5::-2] : [5 3 1]
x[7::-3] : [7 4 1]


### Exercise 1

返回 `x` 中从 index 4 开始的每隔一个元素（即 `x[4], x[6], x[8], ...`）。

<details>
<summary>点击查看答案</summary>

`s = x[4::2]`

</details>

In [19]:
x = np.arange(0, 10)
x[4::2]

array([4, 6, 8])

### Slicing of multidimensional arrays（多维切片）

二维切片写作 `x[row_slice, col_slice]`，两个方向各自独立。

In [20]:
x2 = np.array([[1, 2, 3, 4],
               [5, 6, 7, 8]])
print(x2)
print("x2[0:2, 0:3]  (2 行, 3 列):\n", x2[0:2, 0:3])
print("x2[:, :3]     (所有行, 前 3 列):\n", x2[:, :3])
print("x2[:, ::2]    (所有行, 每隔一列):\n", x2[:, ::2])
print("x2[::-1, ::-1] (两个方向都反转):\n", x2[::-1, ::-1])

[[1 2 3 4]
 [5 6 7 8]]
x2[0:2, 0:3]  (2 行, 3 列):
 [[1 2 3]
 [5 6 7]]
x2[:, :3]     (所有行, 前 3 列):
 [[1 2 3]
 [5 6 7]]
x2[:, ::2]    (所有行, 每隔一列):
 [[1 3]
 [5 7]]
x2[::-1, ::-1] (两个方向都反转):
 [[8 7 6 5]
 [4 3 2 1]]


⚠️ 用整数索引会**降一个维度**，用长度为 1 的切片则保持维度。

In [21]:
print("x2[:, 0]   ->", x2[:, 0], "shape:", x2[:, 0].shape)          # 1-D
print("x2[:, 0:1] ->\n", x2[:, 0:1], "shape:", x2[:, 0:1].shape)    # 2-D

x2[:, 0]   -> [1 5] shape: (2,)
x2[:, 0:1] ->
 [[1]
 [5]] shape: (2, 1)


In [22]:
print(x2[0, :])   # 第 0 行（所有列）
print(x2[0])      # 等价：行访问时末尾的冒号可以省略
print(x2[0, ])    # 也可以写成这样

[1 2 3 4]
[1 2 3 4]
[1 2 3 4]


⚠️ **行访问可以省略列，但列访问不能省略行。** 下面这行是语法错误：

In [23]:
try:
    eval("x2[, 0]")
except SyntaxError as e:
    print("SyntaxError:", e)

# 正确写法：行位置必须写出来（用 ':' 表示所有行）
print("x2[:, 0] ->", x2[:, 0])

SyntaxError: invalid syntax (<string>, line 1)
x2[:, 0] -> [1 5]


## 6. Boolean / conditional indexing（布尔索引 / 条件筛选）

比较运算会生成一个**同形状的布尔数组（mask）**，再用它去筛选或赋值。

In [24]:
x2 = np.array([[1, 2, 3, 4],
               [5, 6, 7, 8]])
print("x2 > 5:\n", x2 > 5)          # 布尔 mask
print("x2[x2 > 5]:", x2[x2 > 5])    # 取出满足条件的元素

x2 > 5:
 [[False False False False]
 [False  True  True  True]]
x2[x2 > 5]: [6 7 8]


In [25]:
x2[x2 > 5] = 0        # 把所有 > 5 的元素改成 0
print(x2)

[[1 2 3 4]
 [5 0 0 0]]


⚠️ 组合多个条件时，必须用 `&`、`|`、`~`，并且**每个条件都要加括号**（不能用 `and` / `or` / `not`）。

In [26]:
x2 = np.array([[1, 2, 3, 4],
               [5, 6, 7, 8]])
print("== 5 :", x2[x2 == 5])
print("!= 5 :", x2[x2 != 5])
print("> 2  :", x2[x2 > 2])
print("2<x<7:", x2[(x2 > 2) & (x2 < 7)])

== 5 : [5]
!= 5 : [1 2 3 4 6 7 8]
> 2  : [3 4 5 6 7 8]
2<x<7: [3 4 5 6]


### Exercise 2

创建与 `x2` 同形状的整数数组 `y`：当 `x2` 对应位置为 0 时 `y` 为 1，否则为 0。

<details>
<summary>点击查看答案</summary>

`y = (x2 == 0).astype(int)`

布尔数组 `True/False` 用 `.astype(int)` 转成 `1/0`。

</details>

In [27]:
x2 = np.array([[1, 2, 3, 4],
               [5, 6, 7, 8]])
x2[x2 > 5] = 0                 # 制造一些 0
print("x2:\n", x2)

y = (x2 == 0).astype(int)      # 布尔 -> 整数
print("y:\n", y)

x2:
 [[1 2 3 4]
 [5 0 0 0]]
y:
 [[0 0 0 0]
 [0 1 1 1]]


## 7. Views vs. copies（视图 vs. 副本）⚠️

**切片得到的是“视图（view）”，不是副本。** 视图和原数组共享同一块内存，改一个会连带改另一个。
这是高频考点，也是很多人踩的坑。

In [28]:
x2 = np.array([[1, 2, 3, 4],
               [5, 6, 7, 8]])
x2_sub = x2[:2, :2]      # 视图，不是副本
x2_sub[0, 0] = 100
print("修改视图后 x2_sub:\n", x2_sub)
print("原数组 x2 也被改了:\n", x2)

修改视图后 x2_sub:
 [[100   2]
 [  5   6]]
原数组 x2 也被改了:
 [[100   2   3   4]
 [  5   6   7   8]]


In [29]:
# 想要真正独立，用 .copy()
x2 = np.array([[1, 2, 3, 4],
               [5, 6, 7, 8]])
x2_copy = x2[:2, :2].copy()
x2_copy[0, 0] = -20
print("副本 x2_copy:\n", x2_copy)
print("原数组 x2 保持不变:\n", x2)

副本 x2_copy:
 [[-20   2]
 [  5   6]]
原数组 x2 保持不变:
 [[1 2 3 4]
 [5 6 7 8]]


> **中文提示**：切片默认是视图（省内存、快）；需要互不影响时显式 `.copy()`。

## 8. Reshaping（改变形状）

### `reshape`：改“形状”，元素个数不变

`reshape` 在可能的情况下返回**视图**。

In [30]:
g = np.arange(1, 10)
grid = g.reshape(3, 3)
print("g     :", g)
print("grid  :\n", grid)

g     : [1 2 3 4 5 6 7 8 9]
grid  :
 [[1 2 3]
 [4 5 6]
 [7 8 9]]


In [31]:
try:
    g.reshape(2, 3)      # 9 个元素塞不进 2x3=6，报错
except ValueError as e:
    print("ValueError:", e)

ValueError: cannot reshape array of size 9 into shape (2,3)


In [32]:
grid[1, 1] = 100
print("grid:\n", grid)
print("g (被改了, 因为 reshape 返回视图):", g)

grid:
 [[  1   2   3]
 [  4 100   6]
 [  7   8   9]]
g (被改了, 因为 reshape 返回视图): [  1   2   3   4 100   6   7   8   9]


### `resize`：改“大小”，可以增删元素（就地修改）

变大时用 0 填充。`refcheck=False` 跳过引用检查（在 notebook 里更省事）。

In [33]:
gg = np.arange(0, 9)
print("before:", gg)
gg.resize(3, 4, refcheck=False)   # 9 -> 12 个元素, 末尾补 0
print("after resize(3,4):\n", gg)

before: [0 1 2 3 4 5 6 7 8]
after resize(3,4):
 [[0 1 2 3]
 [4 5 6 7]
 [8 0 0 0]]


### Adding dimensions with `np.newaxis` / `None`

`None` 和 `np.newaxis` 完全等价，用来**插入一个长度为 1 的新维度**（行向量 / 列向量）。

In [34]:
x = np.array([1, 2, 3])
print("x.shape         :", x.shape)
print("x[None, :].shape:", x[None, :].shape)   # (1, 3) 行向量
print("x[:, None].shape:", x[:, None].shape)   # (3, 1) 列向量
print("x[np.newaxis, :].shape:", x[np.newaxis, :].shape)  # 和 None 相同

x.shape         : (3,)
x[None, :].shape: (1, 3)
x[:, None].shape: (3, 1)
x[np.newaxis, :].shape: (1, 3)


In [35]:
print(x[None, :])     # 行向量
print(x[:, None])     # 列向量
print(x[::2, None])   # 先切片, 再加一个维度

[[1 2 3]]
[[1]
 [2]
 [3]]
[[1]
 [3]]


In [36]:
# 也可以用 reshape 得到行 / 列向量
x = np.arange(0, 6)
print("row   (1, 6):\n", x.reshape(1, 6))
print("column(6, 1):\n", x.reshape(6, 1))

row   (1, 6):
 [[0 1 2 3 4 5]]
column(6, 1):
 [[0]
 [1]
 [2]
 [3]
 [4]
 [5]]


## 9. Concatenating & splitting（拼接与拆分）

`np.concatenate` 把多个数组拼成一个，`axis` 决定沿哪个方向拼。`vstack` / `hstack` 是更方便的封装。

In [37]:
x = np.array([1, 2, 3])
y = np.array([3, 2, 1])
z = [99, 99, 99]

w = np.concatenate([x, y, z])   # list 也会被自动转换
print(w)
print("w.base is None ->", w.base is None)   # 拼接结果是新数组

[ 1  2  3  3  2  1 99 99 99]
w.base is None -> True


In [38]:
grid = np.array([[1, 2, 3],
                 [4, 5, 6]])
print("axis=0 (增加行):\n", np.concatenate([grid, grid], axis=0))
print("axis=1 (增加列):\n", np.concatenate([grid, grid], axis=1))
print("默认 axis=0:\n", np.concatenate([grid, grid]))

axis=0 (增加行):
 [[1 2 3]
 [4 5 6]
 [1 2 3]
 [4 5 6]]
axis=1 (增加列):
 [[1 2 3 1 2 3]
 [4 5 6 4 5 6]]
默认 axis=0:
 [[1 2 3]
 [4 5 6]
 [1 2 3]
 [4 5 6]]


In [39]:
x = np.array([1, 2, 3])
grid = np.array([[9, 8, 7],
                 [6, 5, 4]])
print("vstack (竖直堆叠):\n", np.vstack([x, grid]))

y = np.array([[99],
              [99]])
print("hstack (水平堆叠):\n", np.hstack([grid, y]))

vstack (竖直堆叠):
 [[1 2 3]
 [9 8 7]
 [6 5 4]]
hstack (水平堆叠):
 [[ 9  8  7 99]
 [ 6  5  4 99]]


In [40]:
# split：传入切分点索引列表
x = [1, 2, 3, 99, 99, 3, 2, 1, 1, 2, 3]
x1, x2, x3 = np.split(x, [3, 5])   # 在 index 3 和 5 处切开
print(x1, x2, x3)
print(np.split(x, [3, 5, 8]))

[1 2 3] [99 99] [3 2 1 1 2 3]
[array([1, 2, 3]), array([99, 99]), array([3, 2, 1]), array([1, 2, 3])]


In [41]:
grid = np.arange(16).reshape((2, 8))
print(grid)

upper, lower = np.vsplit(grid, [1])    # 沿行方向切
print("upper:\n", upper)
print("lower:\n", lower)

left, middle, right = np.hsplit(grid, [2, 4])   # 沿列方向切
print("left:\n", left)
print("middle:\n", middle)
print("right:\n", right)

[[ 0  1  2  3  4  5  6  7]
 [ 8  9 10 11 12 13 14 15]]


upper:
 [[0 1 2 3 4 5 6 7]]
lower:
 [[ 8  9 10 11 12 13 14 15]]
left:
 [[0 1]
 [8 9]]
middle:
 [[ 2  3]
 [10 11]]
right:
 [[ 4  5  6  7]
 [12 13 14 15]]


> **中文提示**：`v` = vertical（行方向，axis=0），`h` = horizontal（列方向，axis=1）。

## 10. Iterating over arrays（遍历）

和普通序列一样可以 `for` 循环，但**逐元素 Python 循环很慢**；能用向量化就用向量化，循环只用于确实需要逐项处理的场合。

In [42]:
np.random.seed(0)                       # 固定随机种子, 结果可复现
a = np.random.randint(0, 10, (5, 3))
print(a)

[[5 0 3]
 [3 7 9]
 [3 5 2]
 [4 7 6]
 [8 8 1]]


In [43]:
# 按行遍历
for row in a:
    print(row)

[5 0 3]
[3 7 9]
[3 5 2]
[4 7 6]
[8 8 1]


In [44]:
# 按列遍历：转置后每一行就是原来的每一列
for column in a.T:
    print(column)

[5 3 3 4 8]
[0 7 5 7 8]
[3 9 2 6 1]


In [45]:
# 按下标遍历
for i in range(len(a)):
    print(i, a[i])

0 [5 0 3]
1 [3 7 9]
2 [3 5 2]
3 [4 7 6]
4 [8 8 1]


In [46]:
# enumerate 同时拿到下标和行
print(list(enumerate(a)))
print(dict(enumerate(a)))
for i, row in enumerate(a):
    print("row", i, "is", row)

[(0, array([5, 0, 3], dtype=int32)), (1, array([3, 7, 9], dtype=int32)), (2, array([3, 5, 2], dtype=int32)), (3, array([4, 7, 6], dtype=int32)), (4, array([8, 8, 1], dtype=int32))]
{0: array([5, 0, 3], dtype=int32), 1: array([3, 7, 9], dtype=int32), 2: array([3, 5, 2], dtype=int32), 3: array([4, 7, 6], dtype=int32), 4: array([8, 8, 1], dtype=int32)}
row 0 is [5 0 3]
row 1 is [3 7 9]
row 2 is [3 5 2]
row 3 is [4 7 6]
row 4 is [8 8 1]


In [47]:
# zip 同时遍历多个数组（一一配对）
a2 = a ** 2
for i, j in zip(a, a2):
    print(i, "+", j, "=", i + j)
print(list(zip(a, a2)))

[5 0 3] + [25  0  9] = [30  0 12]
[3 7 9] + [ 9 49 81] = [12 56 90]
[3 5 2] + [ 9 25  4] = [12 30  6]
[4 7 6] + [16 49 36] = [20 56 42]
[8 8 1] + [64 64  1] = [72 72  2]
[(array([5, 0, 3], dtype=int32), array([25,  0,  9], dtype=int32)), (array([3, 7, 9], dtype=int32), array([ 9, 49, 81], dtype=int32)), (array([3, 5, 2], dtype=int32), array([ 9, 25,  4], dtype=int32)), (array([4, 7, 6], dtype=int32), array([16, 49, 36], dtype=int32)), (array([8, 8, 1], dtype=int32), array([64, 64,  1], dtype=int32))]


> **中文提示**：`zip` 以**较短的**序列为准，多出来的部分会被忽略。

## 11. Element-wise operations & dot product（逐元素运算与点积）

`+ - * / ** // %` 全部按**逐元素**进行（两个数组形状相同即可，形状不同时还有“广播”规则）。

In [48]:
x = np.array([1, 2, 3])
y = np.array([4, 5, 6])
print("x      :", x)
print("y      :", y)
print("x + y  :", x + y)
print("x - y  :", x - y)
print("x * y  :", x * y)     # 逐元素相乘, 不是矩阵乘法
print("x / y  :", x / y)
print("x ** 2 :", x ** 2)
print("y // x :", y // x)    # 整除
print("y % x  :", y % x)     # 取余

x      : [1 2 3]
y      : [4 5 6]
x + y  : [5 7 9]
x - y  : [-3 -3 -3]
x * y  : [ 4 10 18]
x / y  : [0.25 0.4  0.5 ]
x ** 2 : [1 4 9]
y // x : [4 2 2]
y % x  : [0 1 0]


**Dot product（点积）：** 两个一维数组相乘再求和。

$$ \begin{bmatrix}x_1 & x_2 & x_3\end{bmatrix} \cdot \begin{bmatrix}y_1 \\ y_2 \\ y_3\end{bmatrix} = x_1y_1 + x_2y_2 + x_3y_3 $$

In [49]:
x = np.array([1, 2, 3])
y = np.array([4, 5, 6])
print("x.dot(y) =", x.dot(y))    # 1*4 + 2*5 + 3*6
print("np.dot   =", np.dot(x, y))

x.dot(y) = 32
np.dot   = 32


⚠️ `*` 是逐元素；**矩阵乘法**要用 `@` 或 `np.dot`。

In [50]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])
print("逐元素 A * B:\n", A * B)
print("矩阵乘法 A @ B:\n", A @ B)

逐元素 A * B:
 [[ 5 12]
 [21 32]]
矩阵乘法 A @ B:
 [[19 22]
 [43 50]]


## 12. Transpose & dtype casting（转置与类型转换）

In [51]:
y = np.array([4, 5, 6])
z = np.array([y, y ** 2])
print(z)
print("z.shape :", z.shape)
print("np.transpose(z):\n", np.transpose(z))
print("z.T:\n", z.T)             # 等价写法
print("z.T.shape:", z.T.shape)
print("z.dtype :", z.dtype)

[[ 4  5  6]
 [16 25 36]]
z.shape : (2, 3)
np.transpose(z):
 [[ 4 16]
 [ 5 25]
 [ 6 36]]
z.T:
 [[ 4 16]
 [ 5 25]
 [ 6 36]]
z.T.shape: (3, 2)
z.dtype : int64


In [52]:
z2 = z.T.astype('f')     # 'f' == float32
print(z2.dtype)
print(z2)

float32
[[ 4. 16.]
 [ 5. 25.]
 [ 6. 36.]]


> **中文提示**：常用 dtype 简写：`'f'`=float32，`'d'`=float64，`'i'`=int32，`'l'`=int64，`'b'`=bool。

## 13. Math / aggregation functions（数学与聚合函数）⚠️

聚合函数把数组“压”成更少的元素。**难点是 `axis` 的含义**：

| axis | 效果 | 记忆 |
|---|---|---|
| `None`（默认） | 整个数组得到一个值 | 全压 |
| `axis=0` | 沿**行**方向压缩 → 每**列**一个值 | 上下压 |
| `axis=1` | 沿**列**方向压缩 → 每**行**一个值 | 左右压 |

> **中文提示**：`axis=0` 就是“把行压掉”，所以结果按**列**排列；`axis=1` 就是“把列压掉”，结果按**行**排列。

In [53]:
a = np.arange(-8, 8, 2).reshape(4, 2)
print(a)
print("a.sum()  =", a.sum())
print("a.sum(0) =", a.sum(0), "  # 每列求和")
print("a.sum(1) =", a.sum(1), "  # 每行求和")

[[-8 -6]
 [-4 -2]
 [ 0  2]
 [ 4  6]]
a.sum()  = -8
a.sum(0) = [-8  0]   # 每列求和
a.sum(1) = [-14  -6   2  10]   # 每行求和


In [54]:
print("max    :", a.max())
print("min    :", a.min())
print("mean   :", a.mean())
print("std    :", a.std())
print("argmax :", a.argmax(), "  # 最大值的展开索引 (flattened)")
print("argmin :", a.argmin())

max    : 6
min    : -8
mean   : -1.0
std    : 4.58257569495584
argmax : 7   # 最大值的展开索引 (flattened)
argmin : 0


In [55]:
# np.func() 形式的常用统计
print("median        :", np.median(a))
print("25th pct      :", np.percentile(a, 25))
print("prod of row 1 :", np.prod(a[1, :]))   # 某一行元素之积

median        : -1.0
25th pct      : -4.5
prod of row 1 : 8


In [56]:
print("any == 0 :", np.any(a == 0))
print("any == 1 :", np.any(a == 1))
print("all < 8  :", np.all(a < 8))
print("count <3 :", np.count_nonzero(a < 3))
print("sum of (a<3)         :", np.sum(a < 3))             # True 当 1
print("sum of (a<3) per row :", np.sum(a < 3, axis=1))     # 每行统计

any == 0 : True
any == 1 : False
all < 8  : True
count <3 : 6
sum of (a<3)         : 6
sum of (a<3) per row : [2 2 2 0]


In [57]:
# 绝对值
print(abs(a))
print(np.abs(a))

[[8 6]
 [4 2]
 [0 2]
 [4 6]]
[[8 6]
 [4 2]
 [0 2]
 [4 6]]


In [58]:
# 三角函数
theta = np.linspace(0, np.pi, 3)
print("theta :", theta)
print("sin   :", np.sin(theta))
print("arcsin:", np.arcsin([-1, 0, 1]))

theta : [0.         1.57079633 3.14159265]
sin   : [0.0000000e+00 1.0000000e+00 1.2246468e-16]
arcsin: [-1.57079633  0.          1.57079633]


In [59]:
# 指数
x = [1, 2, 3]
print("e^x  :", np.exp(x))
print("2^x  :", np.exp2(x))
print("3^x  :", np.power(3, x))

e^x  : [ 2.71828183  7.3890561  20.08553692]
2^x  : [2. 4. 8.]
3^x  : [ 3  9 27]


In [60]:
# 对数
x = [1, 2, 4, 10]
print("ln    :", np.log(x))
print("log2  :", np.log2(x))
print("log10 :", np.log10(x))

ln    : [0.         0.69314718 1.38629436 2.30258509]
log2  : [0.         1.         2.         3.32192809]
log10 : [0.         0.30103    0.60205999 1.        ]


### Exercise 3

对二维数组 `a`，返回**每一行最大值所在的下标**。

<details>
<summary>点击查看答案</summary>

`a.argmax(axis=1)` —— 加上 `axis=1` 后，`argmax` 对每一行分别求最大值的下标。

</details>

In [61]:
a = np.arange(-8, 8, 2).reshape(4, 2)
print(a)
print("每行最大值下标:", a.argmax(axis=1))

[[-8 -6]
 [-4 -2]
 [ 0  2]
 [ 4  6]]


每行最大值下标: [1 1 1 1]


## 14. Sorting（排序）

⚠️ 分清两个东西：

- `np.sort(x)`：**返回排好序的新数组**，`x` 不变。
- `x.sort()`：**就地排序**，`x` 被永久改变。

In [62]:
x = np.array([-4, 2, 3, 1, 5, 1])
print("original         :", x)
print("np.sort (副本)   :", np.sort(x))
print("original 不变    :", x)

x.sort()                       # 就地排序
print("after x.sort()   :", x)
print("reverse           :", x[::-1])

original         : [-4  2  3  1  5  1]
np.sort (副本)   : [-4  1  1  2  3  5]
original 不变    : [-4  2  3  1  5  1]
after x.sort()   : [-4  1  1  2  3  5]
reverse           : [ 5  3  2  1  1 -4]


In [63]:
# argsort：返回“能把它排好序”的下标顺序
x = np.array([-4, 2, 3, 1, 5, 1])
idx = np.argsort(x)
print("x          :", x)
print("argsort    :", idx)
print("x[idx]     :", x[idx])
print("降序 x[idx[::-1]]:", x[idx[::-1]])

x          : [-4  2  3  1  5  1]
argsort    : [0 3 5 1 2 4]
x[idx]     : [-4  1  1  2  3  5]
降序 x[idx[::-1]]: [ 5  3  2  1  1 -4]


> **中文提示**：`argsort` 不改变 `x`，它只是告诉你“按什么下标顺序读 `x` 才是有序的”。所以 `x[argsort(x)]` 得到排序结果，而 `x` 本身没变。

In [64]:
# 2-D 排序：axis 决定沿哪个方向排
y = np.array([-4, 2, 3, 1, 5, 1]).reshape(2, 3)
print("before:\n", y)
y.sort(axis=0)                  # 每一列各自排序
print("sort(axis=0) 每列排序:\n", y)

before:
 [[-4  2  3]
 [ 1  5  1]]
sort(axis=0) 每列排序:
 [[-4  2  1]
 [ 1  5  3]]


In [65]:
y = np.array([-4, 2, 3, 1, 5, 1]).reshape(2, 3)
y.sort(axis=1)                  # 每一行各自排序
print("sort(axis=1) 每行排序:\n", y)

sort(axis=1) 每行排序:
 [[-4  2  3]
 [ 1  1  5]]


## 15. Cheat sheet & common pitfalls（速查表与常见坑）

### 创建
| 想要 | 写法 |
|---|---|
| 从 list 创建 | `np.array([1,2,3])` |
| 全 0 / 全 1 | `np.zeros(shape)` / `np.ones(shape)` |
| 填充固定值 | `np.full(shape, v)` |
| 等差（不含尾） | `np.arange(a, b, step)` |
| 等距（含两端） | `np.linspace(a, b, n)` |

### 形状 / 变形
| 想要 | 写法 |
|---|---|
| 查看形状 | `x.shape` |
| 变形（视图） | `x.reshape(2,3)` |
| 就地改大小 | `x.resize(2,3, refcheck=False)` |
| 加一个维度 | `x[None, :]` 或 `x[:, None]` |
| 转置 | `x.T` / `np.transpose(x)` |
| 拼接 / 拆分 | `np.concatenate` / `np.vstack` / `np.hstack` / `np.split` |

### 常见坑
1. ⚠️ **切片是视图**，改切片会改原数组；要独立用 `.copy()`。
2. ⚠️ **`*` 是逐元素**，矩阵乘法用 `@`。
3. ⚠️ **多条件要用 `&`、`|`、`~`**，并给每个条件加括号。
4. ⚠️ **`stop` 不包含**在切片结果里。
5. ⚠️ **整数数组不能存小数**，赋值小数会被截断；要小数先 `astype(float)`。
6. ⚠️ **`x.sort()` 就地改**，`np.sort(x)` 返回副本。
7. ⚠️ **`axis=0` 压行（按列出结果），`axis=1` 压列（按行出结果）**。
8. ⚠️ **`reshape` 元素总数必须不变**，否则报 `ValueError`。

---

**Done.** 建议现在用 `Kernel → Restart & Run All` 从头跑一遍，再自己动手做一遍三个 Exercise。
遇到和输出不一致的地方，先检查 shape 和 dtype —— 这两个属性几乎能解释 NumPy 里 80% 的“为什么报错”。

<details>
<summary>配套练习：把本节的数组操作和之前的 <code>59486002.ipynb</code> 结合起来</summary>

可以尝试用向量化重写一些循环题，例如“数组元素奇偶计数”：

```python
arr = np.array([1, 4, 3, 11, 7, 8, 15, 10])
odd  = np.sum(arr % 2 == 1)
even = np.sum(arr % 2 == 0)
print("odd :", odd)
print("even:", even)
```

体会一下：不用显式 `for` 循环，也能得到同样结果。

</details>